# 哪些简化真正保留原来的问题？

重复记录与矛盾记录可能只差一个右端。这里先看消去后留下的限制，再处理第一条没有 x 的三元系统。换行是为了找到当前能用的方程，回代是利用最后一条已经只剩一个未知量；每步都要能倒回去。

从新内核按顺序运行；没有滑块时可直接修改参数。计算单元用于核对前面的推理，参考结果放在相应尝试之后。


In [ ]:
first_rhs = 2
second_rhs = 4
leftover = second_rhs - 2 * first_rhs
print(f'第二行减去第一行的 2 倍：0 = {leftover}')

In [ ]:
def check_prediction(value, expected, label):
    if value is None:
        print(label + '：尚未填写，请先在纸上计算，再替换 None。')
        return None
    try:
        proposed = np.asarray(value, dtype=float)
    except (TypeError, ValueError, OverflowError):
        print(label + '：请填写数值、数值列表或数组。')
        return False
    reference = np.asarray(expected, dtype=float)
    if proposed.shape != reference.shape:
        print(label + '：尺寸应为 ' + str(reference.shape) + '，请检查行列。')
        return False
    if not np.all(np.isfinite(proposed)):
        print(label + '：请检查是否除以零或产生了非有限数。')
        return False
    correct = bool(np.allclose(proposed, reference))
    print(label + ('：核对正确。' if correct else '：尚不一致，请回到方程或内积检查。'))
    return correct


结果 `0 = 0` 表示两条方程的解集相同，取交集后仍是整条直线。现在只把第二条改成 `2x + 2y = 5`。左边仍会抵消；先预测右边剩多少，再运行。

In [ ]:
second_rhs = 5
leftover = second_rhs - 2 * first_rhs
print(f'第二行减去第一行的 2 倍：0 = {leftover}')
print('系统有解吗？', leftover == 0)

`0 = 1` 不可能成立，因此两个解集没有共同元素，交集为空。把计算和图像联系起来：重复约束是同一条线，矛盾约束是平行线，唯一解则是相交的两条线。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-1, 4, 300)
cases = [('唯一解', 2 - x, 4 - 2 * x),
         ('无解', 2 - x, 2.5 - x),
         ('无穷多解', 2 - x, 2 - x)]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (title, y1, y2) in zip(axes, cases):
    ax.plot(x, y1, label='first equation')
    ax.plot(x, y2, '--', label='second equation')
    ax.set(title=title, xlabel='x', ylabel='y', xlim=(-1, 4), ylim=(-1, 4))
    ax.grid(alpha=0.2)
axes[0].legend()
fig.tight_layout()
plt.show()

## 三个变量：先找非零主元

系统为 y+z=5、x+y=3、2x+y+z=7。先在纸上说明第一步为什么要交换前两条。

依次记录右端与系数的变化，不只检查最后答案。矩阵记法对应正文下一章；每行仍是一条方程。


In [ ]:
A3 = np.array([[0., 1., 1.], [1., 1., 0.], [2., 1., 1.]])
b3 = np.array([5., 3., 7.])
augmented = np.column_stack((A3, b3))
print('original:\n', augmented)
augmented[[0, 1]] = augmented[[1, 0]]
print('swap first two rows:\n', augmented)
augmented[2] -= 2 * augmented[0]
print('eliminate x:\n', augmented)
augmented[2] += augmented[1]
print('eliminate y:\n', augmented)
assert np.allclose(augmented, [[1,1,0,3], [0,1,1,5], [0,0,2,6]])


## 先预测回代结果，再运行下一格

从最后一行开始，依次求 z、y、x。把你的结果按 `[x,y,z]` 填入 `predicted_solution`；保留 None 不影响从新内核运行全本。错误尺寸也只给反馈。


In [ ]:
predicted_solution = None  # Replace with a list [x, y, z].
check_prediction(predicted_solution, [1, 2, 3], '回代预测')
z3 = augmented[2, 3] / augmented[2, 2]
y3 = (augmented[1, 3] - augmented[1, 2] * z3) / augmented[1, 1]
x3 = (augmented[0, 3] - augmented[0, 1] * y3) / augmented[0, 0]
recovered = np.array([x3, y3, z3])
print('back substitution:', recovered)
assert np.allclose(A3 @ recovered, b3)
assert np.allclose(recovered, [1, 2, 3])


## 有主元，也要先检查矛盾

候选 x=2、y=3，但第三条是 0=1。先判断是否有解，再运行。不能跳过零行的右端。


In [ ]:
inconsistent = np.array([[1.,0.,2.], [0.,1.,3.], [0.,0.,1.]])
zero_left = np.all(np.isclose(inconsistent[:, :-1], 0), axis=1)
contradiction = zero_left & ~np.isclose(inconsistent[:, -1], 0)
print('contradictory rows:', np.flatnonzero(contradiction))
assert contradiction.any()


<details><summary>独立核对与下一步</summary>
换行让 x 的系数为 1；消元后最后一条是 2z=6。回代得到 (1,2,3)。任意一个矛盾行都会使系统无解，即使其他变量已有主元。请另构造一个只有一个自由变量、但没有矛盾的系统，并说明全部解。
</details>


## 回到消元规则

`0 = 0`、`0 = 1` 分别告诉我们什么？第一章中有唯一解的系统，消元后会剩下什么信息？试着先用自己的话回答，再回到教程继续。